In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# as_of = datetime.date(2026, 2, 23)
# start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
# end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

start = NY_tz.localize(datetime.datetime(2026, 4, 9, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 4, 9, 23, 59))

# mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
# pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 30.76it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name,file_date
0,2674560099000000201,470582616,TERM,ETRM,2026-04-09 04:00:00+00:00,None,IR,None,N,False,...,NaN,,,NaN,None,None,QZXR99GN9GQT,NA/Swap Fxd Flt USD,USD-SOFR,2026-04-09
1,2659418548000000301,,NEWT,TRAD,2026-04-09 04:00:05+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
2,2659418867000000101,2659403749000000101,CORR,,2026-04-09 04:00:08+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZJQHSR2XNCJ,NA/Swap OIS SGD,SGD-SORA-OIS Compound,2026-04-09
3,2659419432000000201,2659403749000000101,MODI,TRAD,2026-04-09 04:00:29+00:00,False,IR,None,I,True,...,3.0,,,NaN,None,None,QZJQHSR2XNCJ,NA/Swap OIS SGD,SGD-SORA-OIS Compound,2026-04-09
4,2659419962000000101,,NEWT,TRAD,2026-04-09 04:01:13+00:00,None,IR,None,I,False,...,NaN,,,NaN,None,None,QZJQHSR2XNCJ,NA/Swap OIS SGD,SGD-SORA-OIS Compound,2026-04-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26257,2676491816000000201,,NEWT,TRAD,2026-04-10 03:57:11+00:00,None,IR,None,I,False,...,NaN,,,NaN,None,None,QZP63MDJTXMG,NA/Swap Fxd Flt TWD,TWD-TAIBOR-Reuters,2026-04-10
26258,2676481671000000101,,NEWT,TRAD,2026-04-10 03:57:58+00:00,None,IR,None,N,False,...,NaN,,,NaN,None,None,QZLZ5C727TTM,NA/Swap Fxd Fxd JPY USD,N/A,2026-04-10
26259,2676481235000000101,,NEWT,TRAD,2026-04-10 03:58:21+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZV61TRS4HD9,NA/Swap OIS JPY,JPY-TONA-OIS-COMPOUND,2026-04-10
26260,2676481417000000101,,NEWT,TRAD,2026-04-10 03:58:46+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound,2026-04-10


In [21]:
# df["Event timestamp"] = df["Event timestamp"].astype(str)
# df["Execution Timestamp"] = df["Execution Timestamp"].astype(str)
# df.to_csv(r"C:\Users\chris\clee\ARBS\notebooks\sdr\april_fomc_dated_sdr_trades.csv", index=False)	

In [3]:
# df[(df["Effective Date"].dt.date == datetime.date(2026, 4, 28)) & (df["Expiration Date"].dt.date == datetime.date(2026, 6, 16))]

In [7]:
df[df["Dissemination Identifier"] == "2672209446000001201"].iloc[0].to_dict()

{'Dissemination Identifier': '2672209446000001201',
 'Original Dissemination Identifier': '',
 'Action type': 'NEWT',
 'Event type': 'TRAD',
 'Event timestamp': Timestamp('2026-04-09 16:54:17+0000', tz='UTC'),
 'Amendment indicator': None,
 'Asset Class': 'IR',
 'Product name': None,
 'Cleared': 'I',
 'Mandatory clearing indicator': True,
 'Execution Timestamp': Timestamp('2026-04-09 16:54:17+0000', tz='UTC'),
 'Effective Date': Timestamp('2026-04-29 00:00:00'),
 'Expiration Date': Timestamp('2026-06-17 00:00:00'),
 'Maturity date of the underlier': None,
 'Non-standardized term indicator': None,
 'Platform identifier': 'TWSF',
 'Prime brokerage transaction indicator': False,
 'Block trade election indicator': True,
 'Large notional off-facility swap election indicator': None,
 'Notional amount-Leg 1': '7,500,000,000+',
 'Notional amount-Leg 2': '7,500,000,000+',
 'Notional currency-Leg 1': 'USD',
 'Notional currency-Leg 2': 'USD',
 'Notional quantity-Leg 1': None,
 'Notional quantity-